# SHAP Tabular Attribution Analysis
Chạy trên Colab T4 GPU. Kết quả: `shap_tabular.png` + `shap_summary.png`

**Trước khi chạy:**
1. Upload thư mục `models/best_model/` lên Google Drive → `MyDrive/deep_sentiment/best_model/`
2. Upload file `data/processed/test.parquet` lên Drive → `MyDrive/deep_sentiment/test.parquet`
3. Runtime → Change runtime type → **T4 GPU**

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo (public) — lấy source code mới nhất
import os
if not os.path.exists('/content/deep-social-sentiment-analysis'):
    !git clone https://github.com/nhiney/deep-social-sentiment-analysis.git
os.chdir('/content/deep-social-sentiment-analysis')
!git pull origin main
print('Working dir:', os.getcwd())

In [ ]:
# 3. Install dependencies
!pip install shap joblib transformers sentencepiece pyarrow -q
print('Done installing')

In [ ]:
# 4. Copy model + data từ Drive
import shutil, os

DRIVE_BASE = '/content/drive/MyDrive/deep_sentiment'

# Model checkpoint
os.makedirs('models/best_model', exist_ok=True)
for fname in ['pytorch_model.bin', 'config.json', 'tab_preprocessor.joblib']:
    src = f'{DRIVE_BASE}/best_model/{fname}'
    dst = f'models/best_model/{fname}'
    if not os.path.exists(dst):
        print(f'Copying {fname}...')
        shutil.copy2(src, dst)
    else:
        print(f'{fname} already exists, skipping')

# Test data
os.makedirs('data/processed', exist_ok=True)
test_dst = 'data/processed/test.parquet'
if not os.path.exists(test_dst):
    print('Copying test.parquet...')
    shutil.copy2(f'{DRIVE_BASE}/test.parquet', test_dst)
else:
    print('test.parquet already exists, skipping')

print('\nFiles ready:')
!ls -lh models/best_model/ && ls -lh data/processed/test.parquet

In [ ]:
# 5. Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 6. Chạy SHAP — full params (BG=30, N=80, nsamples=100)
# GPU: ~15-20 phút. Nếu muốn test nhanh đổi SHAP_N=20
import subprocess, sys

env = os.environ.copy()
env['SHAP_BG']      = '30'
env['SHAP_N']       = '80'
env['SHAP_SAMPLES'] = '100'

result = subprocess.run(
    [sys.executable, '-m', 'scripts.run_shap_analysis'],
    env=env,
    capture_output=False   # in trực tiếp ra Colab output
)
print('\nExit code:', result.returncode)

In [ ]:
# 7. Preview figures trong Colab
from IPython.display import Image, display
print('=== SHAP Heatmap ===')
display(Image('reports/figures/shap_tabular.png'))
print('=== Feature Importance Bar ===')
display(Image('reports/figures/shap_summary.png'))

In [ ]:
# 8a. Download figures trực tiếp về máy
from google.colab import files
files.download('reports/figures/shap_tabular.png')
files.download('reports/figures/shap_summary.png')
files.download('reports/shap_values.npy')
print('Downloaded 3 files')

In [ ]:
# 8b. (Tuỳ chọn) Backup lên Drive
import shutil
os.makedirs(f'{DRIVE_BASE}/shap_output', exist_ok=True)
for f in ['reports/figures/shap_tabular.png',
          'reports/figures/shap_summary.png',
          'reports/shap_values.npy']:
    shutil.copy2(f, f'{DRIVE_BASE}/shap_output/')
    print(f'Backed up: {f}')